# Meta-Labeling with sklearn — Exercise 3.5b

Split from the original AFML 3.2 notebook; the labeling step (Exercise 3.5) lives in `AFML 3.2.1 - Triple Barrier Labeling.ipynb`.

This notebook is self-contained: it re-reads the data and rebuilds the labels before training.

The primary-model-only comparison is in `AFML 3.2.2b - Primary Model Only.ipynb`.

As seen in the labeling exercise, only ~48.04% of the sample was labeled 1.
Hence precision 1.0 = 0.48 (48% of the sample is relevant), while recall = 1 means fully correct (based on the 48% sample).



In [ ]:
import numpy as np
import pandas as pd
import cqrlib as rs
import matplotlib.pyplot as plt

%matplotlib inline

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from statsmodels.tsa.arima.model import ARIMA  # ARMA was removed in statsmodels 0.13

p = print

dollar = pd.read_csv('../sample-data/dollar_bars.csv', 
                 sep=',', 
                 header=0, 
                 parse_dates = True, 
                 index_col=['date_time'])


In [ ]:
# fraction form keeps the original pipeline (dollar) intact; moment form goes to a parallel frame
dollar['ewm'], dollar['upper'], dollar['lower'], dollar['std'] = rs.bband_frac(dollar)
dollar_m = dollar.copy()
dollar_m['ewm'], dollar_m['upper'], dollar_m['lower'], dollar_m['std'] = rs.bband_std(dollar_m)

dollar['side'] = np.nan

upper = dollar[dollar['upper'] < dollar['close']] # short signal
lower = dollar[dollar['lower'] > dollar['close']] # long signal

p("Num of times upper limit touched: {0}\nNum of times lower limit touched: {1}"
  .format(len(upper), 
          len(lower)))

# Recall white test as a benchmark and until this stage we filtered all those which did not meet min return
dollar = rs.side_pick(dollar)
dollar.dropna(inplace= True)
dollar['side'].value_counts()


In [ ]:
# ---- compare the two band forms ----
def band_summary(df, label):
    return pd.Series({
        'upper touches': (df['close'] >= df['upper']).sum(),
        'lower touches': (df['close'] <= df['lower']).sum(),
        'mean width': df['std'].mean(),
        'width / price %': (df['std'] / df['close']).mean() * 100,
    }, name=label)

pd.DataFrame([band_summary(dollar, 'bband_frac'),
              band_summary(dollar_m, 'bband_std')])

# compare on the full series so both bands share the same window (dollar is dropna'd)
frac = dollar_m[['close']].copy()
frac['ewm'], frac['upper'], frac['lower'], frac['std'] = rs.bband_frac(frac)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

win = slice(-400, None)  # last 400 bars
ax = axes[0]
ax.plot(frac['close'].iloc[win], lw=1, label='close')
ax.plot(frac['upper'].iloc[win], c='C1', alpha=.6)
ax.plot(frac['lower'].iloc[win], c='C1', alpha=.6, label='frac bands')
ax.plot(dollar_m['upper'].iloc[win], c='C2', alpha=.6, ls='--')
ax.plot(dollar_m['lower'].iloc[win], c='C2', alpha=.6, ls='--', label='std bands')
ax.legend(); ax.set_title('Band forms over the last 400 bars')

ax = axes[1]
ratio = dollar_m['std'] / frac['std']  # ~ realized vol / 0.001
ax.plot(ratio.index, ratio, lw=.7)
ax.set_title('width ratio (std form / frac form)')
plt.tight_layout()


In [ ]:
copy_dollar = dollar.copy() # make a back copy to be used in the feature engineering below

copy_dollar # up till this point the below dataframe should look like this, before tri_bar func. This is our primary model.


In [ ]:
# rebuild the baseline labels on the primary model for the classification-report notes
d_vol = rs.vol(dollar['close'], span0 = 50)

# d_vol is a return (~0.55%); cs_filter diffs are price points, so scale by price level
events = rs.cs_filter(dollar['close'], 
                    limit = d_vol.mean() * dollar['close'].mean())

vb = rs.vert_barrier(data = dollar['close'], 
                 events = events, 
                 period = 'days', 
                 freq = 1)

tb = rs.tri_barrier(data = dollar['close'], 
                events = events, 
                trgt = d_vol, 
                min_req = 0.002, 
                num_threads = 3, 
                ptSl = [0,2], # change ptSl into [0,2]
                t1 = vb, 
                side = dollar['side'])

m_label = rs.meta_label(data = dollar['close'],
                      events = tb,
                      drop = False)

# this func can be found under Tools/stats_rpt
forecast = rs.report_matrix(actual_data = m_label, 
                            prediction_data = None,
                            ROC = None)


The below function report_matrix is what we have till date using both primary (bband func) and secondary model (tri_bar func).

#### Classification Report

As seen in previous example, only 48.0455% was labeled 1. 

Hence precision 1.0 = 0.48 (48.0455% of the sample is relevant). It's basically ML's way of saying are these "features" relevant when tested.

While recall = 1 means fully correct (based on the 48% sample). In the case where ML model is fitted, this result will mean the percentage of "correct" label was chosen. In short, is the ML model reliability in True positive identification based on given sample.

#### Confusion Matrix

8001 = False Positive (51.95%)
7399 = True Positive (48.0455%)

#### Accuracy Score

Is a mere reflection of True Positive, which again is 48.0455%


In [ ]:
# drop redundant columns and keep crossing moving avaerages
pri_dollar = copy_dollar.drop(['open', 'high', 'low', 'cum_vol', 'cum_dollar', 'cum_ticks'], axis = 1)

# include original volatility
pri_dollar['volatility'] = rs.vol(pri_dollar.close, span0 = 50)

pri_dollar


In [ ]:
# Optional: getting stationarity feature
pri_dollar['log_price'] = pri_dollar.close.apply(np.log)
pri_dollar['log_return'] = pri_dollar.log_price.diff()

cs_log = pri_dollar.log_price.diff().dropna().to_frame()
pri_dollar['stationary'] = rs.fracDiff_FFD(data = cs_log, d = 1.99999889 , thres = 1e-5)

rs.unit_root(pri_dollar['stationary'].dropna()) #check for stationarity


In [ ]:
pri_dollar.dropna(inplace = True)


In [ ]:
# autocorrelation residual feature, we will add AR features up to 2 lags
pri_dollar['ar_0'] = ARIMA(pri_dollar['stationary'], order=(0,0,0)).fit().resid
pri_dollar['ar_1'] = ARIMA(pri_dollar['stationary'], order=(1,0,0)).fit().resid
pri_dollar['ar_2'] = ARIMA(pri_dollar['stationary'], order=(2,0,0)).fit().resid


In [ ]:
# final dataset
secondary_dollar = pri_dollar.copy()


In [ ]:
# Now we run all the steps to complete labels, to train random forest.
# we will use both primary & secondary model

# volatility is a return; cs_filter diffs are price points, so scale by price level
events0 = rs.cs_filter(secondary_dollar['close'], 
                    limit = secondary_dollar['volatility'].mean() * secondary_dollar['close'].mean())

vb0 = rs.vert_barrier(data = secondary_dollar['close'], 
                 events = events0, 
                 period = 'days', 
                 freq = 1)

tb0 = rs.tri_barrier(data = secondary_dollar['close'], 
                events = events0, 
                trgt = secondary_dollar['volatility'], 
                min_req = 0.002, 
                num_threads = 3, 
                ptSl = [0,2], # change ptSl into [0,2]
                t1 = vb0, 
                side = secondary_dollar['side'])

m_label0 = rs.meta_label(data = secondary_dollar['close'],
                      events = tb0,
                      drop = 0.05)

m_label0


In [ ]:
m_label0['bin'].value_counts() 

# we still get back the same count. This is correct. 
# Tri_bar func is to calculate if vert_bar was triggered and consolidates the target.
# while label will check which are the ones that hitted vertical barriers or non-profitable will be label 0


In [ ]:
# At this stage you may wish to run Grid search CV, but I'm skipping that.

# n_estimators, max_depth, c_random_state = 500, 7, 42
# n_estimators, max_depth, c_random_state = 100, 5, 42
n_estimators, max_depth, c_random_state = 50, 5, 42

# Random Forest Model
rf = RandomForestClassifier(max_depth=max_depth, 
                            n_estimators=n_estimators,
                            criterion='entropy', 
                            class_weight = None, #This will be cover in next few chapters
                            random_state=c_random_state)

X = secondary_dollar.reindex(m_label0.index) # this dataframe only contain all our features
y = m_label0['bin']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

rf.fit(X_train, y_train.values.ravel())


In [ ]:
# Performance Metrics
y_prob = rf.predict_proba(X_train)[:, 1] #here we are only interested in True positive
y_pred = rf.predict(X_train)

p('Matrix training report for primary model & secondary model\n')

rs.report_matrix(actual_data = y_train, # we need to use our train data from train_test_split
                 prediction_data = y_pred, 
                 ROC = y_prob)


In [ ]:
# Meta-label
# Performance Metrics
y_prob = rf.predict_proba(X_test)[:, 1] #here we are only interested in True positive
y_pred = rf.predict(X_test)

p('Matrix test report for primary model & secondary model\n')

rs.report_matrix(actual_data = y_test, 
                 prediction_data = y_pred, 
                 ROC = y_prob)

rs.feat_imp(rf, X)


In [ ]:
# ---- full-pipeline comparison: band forms vs MA-cross vs fracdiff factor ----
def band_side(band, window=21, width=None):
    """Side labels from a band form: -1 upper touch (short), +1 lower touch (long)."""
    def apply(data):
        if width is None:
            data['ewm'], data['upper'], data['lower'], data['std'] = band(data, window=window)
        else:
            data['ewm'], data['upper'], data['lower'], data['std'] = band(data, window=window, width=width)
        data['side'] = np.nan
        return rs.side_pick(data)
    return apply

def ma_cross_side(fast=5, slow=20):
    """Side labels from MA-crossing events: +1 long / -1 short at the crossing bar."""
    def apply(data):
        data['side'] = rs.get_ma_crossing_signals(
            data['close'], fast_window=fast, slow_window=slow).reindex(data.index)
        return data
    return apply

def fracdiff_side(d=0.5, thres=1e-5, k=1.0):
    """Mean-reversion side on a fractionally-differentiated log price:
    short when z > k*sigma, long when z < -k*sigma (revert to 0)."""
    def apply(data):
        cs = data['close'].apply(np.log).diff().dropna().to_frame()
        z = rs.fracDiff_FFD(data=cs, d=d, thres=thres).iloc[:, 0]
        sigma = z.std()
        side = pd.Series(np.nan, index=data.index)
        side.loc[z.index] = np.where(z > k * sigma, -1.0,
                                     np.where(z < -k * sigma, 1.0, np.nan))
        data['side'] = side
        return data
    return apply

def run_pipeline(side_fn, label, min_req=0.002, drop=0.05, ptSl=[0, 2],
                 t1_freq=1, cs_scale=1.0, thr=0.5, n_estimators=100, max_depth=5):
    data = pd.read_csv('../sample-data/dollar_bars.csv', sep=',', header=0,
                       parse_dates=True, index_col=['date_time'])
    data = side_fn(data)
    data.dropna(inplace=True)

    pri = data.drop(['open', 'high', 'low', 'cum_vol', 'cum_dollar', 'cum_ticks'], axis=1)
    pri['volatility'] = rs.vol(pri.close, span0=50)
    pri['log_price'] = pri.close.apply(np.log)
    pri['log_return'] = pri.log_price.diff()
    cs_log = pri.log_price.diff().dropna().to_frame()
    pri['stationary'] = rs.fracDiff_FFD(data=cs_log, d=1.99999889, thres=1e-5)
    pri.dropna(inplace=True)
    pri['ar_0'] = ARIMA(pri['stationary'], order=(0, 0, 0)).fit().resid
    pri['ar_1'] = ARIMA(pri['stationary'], order=(1, 0, 0)).fit().resid
    pri['ar_2'] = ARIMA(pri['stationary'], order=(2, 0, 0)).fit().resid
    secondary = pri.copy()

    # cs_scale < 1 -> lower cusum threshold -> denser event sampling
    events0 = rs.cs_filter(secondary['close'],
                           limit=cs_scale * secondary['volatility'].mean() * secondary['close'].mean())
    vb0 = rs.vert_barrier(data=secondary['close'], events=events0,
                          period='days', freq=t1_freq)
    tb0 = rs.tri_barrier(data=secondary['close'], events=events0,
                         trgt=secondary['volatility'], min_req=min_req, num_threads=3,
                         ptSl=ptSl, t1=vb0, side=secondary['side'])
    m_label0 = rs.meta_label(data=secondary['close'], events=tb0, drop=drop)

    # share of events where neither PT nor SL was hit before t1 (vbarrier-resolved)
    t1s = tb0['t1'].fillna(secondary['close'].index[-1])
    pt0 = ptSl[0] * tb0['trgt'] if ptSl[0] > 0 else None
    sl0 = -ptSl[1] * tb0['trgt'] if ptSl[1] > 0 else None
    n_vbar = 0
    for loc, t1 in t1s.items():
        df0 = (secondary['close'].loc[loc:t1] / secondary['close'].at[loc] - 1) * secondary['side'].at[loc]
        hit = (df0 > pt0[loc]).any() if pt0 is not None else False
        hit = hit or ((df0 < sl0[loc]).any() if sl0 is not None else False)
        n_vbar += int(not hit)
    vbar_pct = n_vbar / len(t1s)

    X = secondary.reindex(m_label0.index)
    y = m_label0['bin']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)
    rf = RandomForestClassifier(max_depth=max_depth, n_estimators=n_estimators,
                                criterion='entropy', random_state=42)
    rf.fit(X_train, y_train.values.ravel())

    y_prob = rf.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= thr).astype(int)   # thr>0.5 -> fewer, higher-confidence bets
    p_, r_, f1, _ = precision_recall_fscore_support(y_test, y_pred, labels=[0, 1])
    return pd.Series({
        'accuracy': accuracy_score(y_test, y_pred),
        'precision_1': p_[1], 'recall_1': r_[1],
        'f1_0': f1[0], 'f1_1': f1[1],
        'n_labeled': len(y),
        'n_bets': int(y_pred.sum()),
        'vbar_pct': vbar_pct,
    }, name=label)

pd.DataFrame([
    # asymmetric baseline (ptSl=[0,2], no profit-take barrier)
    run_pipeline(band_side(rs.bband_frac), 'frac (default)'),
    run_pipeline(band_side(rs.bband_std, width=0.5), 'std w=0.5'),
    run_pipeline(band_side(rs.bband_std, width=1.0), 'std w=1.0'),
    run_pipeline(band_side(rs.bband_std, width=2.0), 'std w=2.0'),
    run_pipeline(ma_cross_side(), 'ma_cross 5-20'),
    run_pipeline(ma_cross_side(fast=5, slow=50), 'ma_cross 5-50'),
    run_pipeline(fracdiff_side(d=0.5, k=0.5), 'fracdiff d=0.5 k=0.5'),
    run_pipeline(fracdiff_side(d=1.0, k=0.5), 'fracdiff d=1.0 k=0.5'),
    # d=0.3 needs a looser weight truncation (thres=1e-3), otherwise FFD is very slow
    run_pipeline(fracdiff_side(d=0.3, thres=1e-3, k=0.5), 'fracdiff d=0.3 k=0.5'),
    # symmetric barriers (ptSl=[2,2]): real profit-take at +2σ, stop at -2σ
    run_pipeline(band_side(rs.bband_frac), 'frac ptSl=[2,2]', ptSl=[2, 2]),
    run_pipeline(band_side(rs.bband_std, width=0.5), 'std w=0.5 ptSl=[2,2]', ptSl=[2, 2]),
    run_pipeline(ma_cross_side(), 'ma_cross 5-20 ptSl=[2,2]', ptSl=[2, 2]),
    run_pipeline(fracdiff_side(d=1.0, k=0.5), 'fracdiff d=1.0 ptSl=[2,2]', ptSl=[2, 2]),
    # near profit-take, far stop (ptSl=[0.5,2]): high-frequency, small-win labels
    run_pipeline(band_side(rs.bband_frac), 'frac ptSl=[0.5,2]', ptSl=[0.5, 2]),
    run_pipeline(band_side(rs.bband_std, width=0.5), 'std w=0.5 ptSl=[0.5,2]', ptSl=[0.5, 2]),
    run_pipeline(ma_cross_side(), 'ma_cross 5-20 ptSl=[0.5,2]', ptSl=[0.5, 2]),
    run_pipeline(fracdiff_side(d=1.0, k=0.5), 'fracdiff d=1.0 ptSl=[0.5,2]', ptSl=[0.5, 2]),
    # near profit-take, moderate stop (ptSl=[0.5,1])
    run_pipeline(band_side(rs.bband_frac), 'frac ptSl=[0.5,1]', ptSl=[0.5, 1]),
    run_pipeline(band_side(rs.bband_std, width=0.5), 'std w=0.5 ptSl=[0.5,1]', ptSl=[0.5, 1]),
    run_pipeline(ma_cross_side(), 'ma_cross 5-20 ptSl=[0.5,1]', ptSl=[0.5, 1]),
    run_pipeline(fracdiff_side(d=1.0, k=0.5), 'fracdiff d=1.0 ptSl=[0.5,1]', ptSl=[0.5, 1]),
    # longer vertical barrier: fewer vbarrier-resolved (coin-flip) labels
    run_pipeline(band_side(rs.bband_frac), 'frac t1=5d', t1_freq=5),
    run_pipeline(band_side(rs.bband_frac), 'frac t1=10d', t1_freq=10),
    run_pipeline(band_side(rs.bband_std, width=0.5), 'std w=0.5 t1=5d', t1_freq=5),
    run_pipeline(ma_cross_side(), 'ma_cross 5-20 t1=5d', t1_freq=5),
    # denser event sampling: lower cusum threshold -> more observations
    run_pipeline(band_side(rs.bband_frac), 'frac cs x0.5', cs_scale=0.5),
    run_pipeline(band_side(rs.bband_frac), 'frac cs x0.25', cs_scale=0.25),
    # higher meta-model decision threshold: fewer, higher-confidence bets
    run_pipeline(band_side(rs.bband_frac), 'frac thr=0.6', thr=0.6),
    run_pipeline(band_side(rs.bband_frac), 'frac thr=0.7', thr=0.7),
    # combined: denser sampling + stricter threshold
    run_pipeline(band_side(rs.bband_frac), 'frac cs x0.5 thr=0.7', cs_scale=0.5, thr=0.7),
    # same two levers on the best config (ma_cross t1=5d): both hurt
    run_pipeline(ma_cross_side(), 'ma_cross t1=5d cs x0.5', t1_freq=5, cs_scale=0.5),
    run_pipeline(ma_cross_side(), 'ma_cross t1=5d cs x0.25', t1_freq=5, cs_scale=0.25),
    run_pipeline(ma_cross_side(), 'ma_cross t1=5d thr=0.6', t1_freq=5, thr=0.6),
    run_pipeline(ma_cross_side(), 'ma_cross t1=5d thr=0.7', t1_freq=5, thr=0.7),
    run_pipeline(ma_cross_side(), 'ma_cross t1=5d cs x0.5 thr=0.7', t1_freq=5, cs_scale=0.5, thr=0.7),
]).round(3)


In [ ]:
# ---- visualize the signal sources and their partitions ----
win = slice(-800, None)

d = pd.read_csv('../sample-data/dollar_bars.csv', sep=',', header=0,
                parse_dates=True, index_col=['date_time'])
d_frac = d.copy()
d_frac['ewm'], d_frac['upper'], d_frac['lower'], d_frac['std'] = rs.bband_frac(d_frac)
d_std = d.copy()
d_std['ewm'], d_std['upper'], d_std['lower'], d_std['std'] = rs.bband_std(d_std)

# panels 1-3 share the datetime x-axis; panel 4 gets its own categorical axis
# (bars on a datetime axis would cram into the epoch corner)
fig, axes = plt.subplots(3, 1, figsize=(15, 13), sharex=True)

# 1) band touches: frac fires almost everywhere, std 2sigma rarely
ax = axes[0]
w = d_frac.iloc[win]
ax.plot(w['close'], lw=.7, label='close')
ax.plot(w['upper'], c='C1', alpha=.5, lw=.7, label='frac bands')
ax.plot(w['lower'], c='C1', alpha=.5, lw=.7)
ax.plot(d_std['upper'].iloc[win], c='C2', alpha=.5, ls='--', lw=.7, label='std 2σ bands')
ax.plot(d_std['lower'].iloc[win], c='C2', alpha=.5, ls='--', lw=.7)
short = w[w['close'] >= w['upper']].index
long = w[w['close'] <= w['lower']].index
ax.scatter(short, w.loc[short, 'close'], c='r', s=7, marker='v', alpha=.6, label='frac short')
ax.scatter(long, w.loc[long, 'close'], c='g', s=7, marker='^', alpha=.6, label='frac long')
ax.legend(fontsize=8)
ax.set_title('1) band touches: frac (0.1%) fires on most bars, std 2σ almost never')

# 2) MA cross 5-20: only crossing bars get a side (sparse)
ax = axes[1]
fast = d['close'].rolling(5).mean()
slow = d['close'].rolling(20).mean()
w = d.iloc[win]
ax.plot(w['close'], lw=.7, label='close')
ax.plot(fast.iloc[win], lw=.7, c='C1', label='MA5')
ax.plot(slow.iloc[win], lw=.7, c='C2', label='MA20')
sigs = rs.get_ma_crossing_signals(d['close'], fast_window=5, slow_window=20)
sigs_w = sigs.reindex(w.index).dropna()
ax.scatter(sigs_w.index, w.loc[sigs_w.index, 'close'],
           c=np.where(sigs_w.values > 0, 'g', 'r'), s=25, zorder=5)
ax.legend(fontsize=8)
ax.set_title('2) MA cross 5-20: side only at crossing bars (sparse)')

# 3) fractionally-differentiated log price: lower d = more memory, smoother
ax = axes[2]
cs = d['close'].apply(np.log).diff().dropna().to_frame()
for dd, c in [(0.3, 'C0'), (0.5, 'C1'), (1.0, 'C2')]:
    z = rs.fracDiff_FFD(data=cs, d=dd, thres=1e-3).iloc[:, 0].reindex(w.index)
    ax.plot(z, lw=.7, c=c, label=f'fracdiff d={dd}')
ax.axhline(0, c='k', lw=.5)
ax.legend(fontsize=8)
ax.set_title('3) fracdiff z-series: d=0.3 smooth & long-memory, d=1.0 noisier')

# 4) signal counts per source (log scale: frac ~16k vs ma_cross ~100)
ax4 = fig.add_subplot(4, 1, 4)
labels, long_n, short_n = [], [], []
u = (d_frac['close'] >= d_frac['upper']).sum()
l = (d_frac['close'] <= d_frac['lower']).sum()
labels.append('frac'); short_n.append(u); long_n.append(l)
u = (d_std['close'] >= d_std['upper']).sum()
l = (d_std['close'] <= d_std['lower']).sum()
labels.append('std 2σ'); short_n.append(u); long_n.append(l)
sigs = rs.get_ma_crossing_signals(d['close'])
labels.append('ma_cross'); long_n.append((sigs > 0).sum()); short_n.append((sigs < 0).sum())
z = rs.fracDiff_FFD(data=cs, d=0.5, thres=1e-3).iloc[:, 0]
sigma = z.std()
labels.append('fracdiff d=.5'); long_n.append((z < -0.5 * sigma).sum()); short_n.append((z > 0.5 * sigma).sum())

x = np.arange(len(labels))
ax4.bar(x - .2, short_n, .4, label='short signals', color='r')
ax4.bar(x + .2, long_n, .4, label='long signals', color='g')
ax4.set_xticks(x); ax4.set_xticklabels(labels)
ax4.set_yscale('log')
ax4.legend(fontsize=8)
ax4.set_title('4) signal counts per source (log scale)')

plt.tight_layout(h_pad=2.5)


In [ ]:
# ---- RF hyperparameter search on band signals (time-series CV) ----
# note: cqrlib PurgedKFold indexes events positionally and breaks on DatetimeIndex,
# so we use sklearn TimeSeriesSplit (same future-free property)
def build_data(side_fn, min_req=0.002, drop=0.05):
    data = pd.read_csv('../sample-data/dollar_bars.csv', sep=',', header=0,
                       parse_dates=True, index_col=['date_time'])
    data = side_fn(data)
    data.dropna(inplace=True)

    pri = data.drop(['open', 'high', 'low', 'cum_vol', 'cum_dollar', 'cum_ticks'], axis=1)
    pri['volatility'] = rs.vol(pri.close, span0=50)
    pri['log_price'] = pri.close.apply(np.log)
    pri['log_return'] = pri.log_price.diff()
    cs_log = pri.log_price.diff().dropna().to_frame()
    pri['stationary'] = rs.fracDiff_FFD(data=cs_log, d=1.99999889, thres=1e-5)
    pri.dropna(inplace=True)
    pri['ar_0'] = ARIMA(pri['stationary'], order=(0, 0, 0)).fit().resid
    pri['ar_1'] = ARIMA(pri['stationary'], order=(1, 0, 0)).fit().resid
    pri['ar_2'] = ARIMA(pri['stationary'], order=(2, 0, 0)).fit().resid
    secondary = pri.copy()

    events0 = rs.cs_filter(secondary['close'],
                           limit=secondary['volatility'].mean() * secondary['close'].mean())
    vb0 = rs.vert_barrier(data=secondary['close'], events=events0, period='days', freq=1)
    tb0 = rs.tri_barrier(data=secondary['close'], events=events0,
                         trgt=secondary['volatility'], min_req=min_req, num_threads=3,
                         ptSl=[0, 2], t1=vb0, side=secondary['side'])
    m_label0 = rs.meta_label(data=secondary['close'], events=tb0, drop=drop)
    return secondary.reindex(m_label0.index), m_label0['bin']

def tune_rf(side_fn, label):
    X, y = build_data(side_fn)
    split = int(len(X) * 0.7)
    X_tr, X_te, y_tr, y_te = X.iloc[:split], X.iloc[split:], y.iloc[:split], y.iloc[split:]

    pipe = Pipeline([('clf', RandomForestClassifier(criterion='entropy', random_state=42))])
    param_grid = {'clf__n_estimators': [50, 100, 200], 'clf__max_depth': [3, 5, 7]}
    gs = GridSearchCV(pipe, param_grid, scoring='f1',
                      cv=TimeSeriesSplit(n_splits=3), n_jobs=-1)
    gs.fit(X_tr, y_tr)
    clf = gs.best_estimator_.named_steps['clf']

    y_pred = gs.best_estimator_.predict(X_te)
    p_, r_, f1, _ = precision_recall_fscore_support(y_te, y_pred, labels=[0, 1])
    return pd.Series({
        'best_n_estimators': clf.n_estimators,
        'best_max_depth': clf.max_depth,
        'cv_f1': gs.best_score_,
        'accuracy': accuracy_score(y_te, y_pred),
        'f1_1': f1[1],
        'n_labeled': len(y),
    }, name=label)

pd.DataFrame([
    tune_rf(band_side(rs.bband_frac), 'frac (default)'),
    tune_rf(band_side(rs.bband_std, width=0.5), 'std w=0.5'),
    tune_rf(band_side(rs.bband_std, width=1.0), 'std w=1.0'),
]).round(3)


## Band-form comparison — conclusion

Running the full pipeline (bands → side → features → labels → RF) for both band forms shows that the band *form* is not what drives model quality — the **bandwidth → signal count → sample size** chain is.

- At matched sample sizes the two forms converge: `bband_frac` (1035 labels) vs `bband_std w=0.5` (1024 labels) give nearly identical metrics (accuracy 0.521 vs 0.513, f1_1 0.498 vs 0.519).
- Narrowing the σ-multiple width increases signal count: w=0.5 → 1024, w=1.0 → 665, w=1.5 → 291, w=2.0 → 45 labels.
- The apparent improvement at w=2.0 (accuracy 0.643, recall_1 = 1.0) is a small-sample artifact: ~45 labels ≈ 13 test rows, where the model effectively guesses class 1.

### MA-cross factor

Swapping the band-touch side for the MA-cross factor (`get_ma_crossing_signals`) makes the meta-model **fail to beat the 0.5 random baseline**: accuracy 0.371 (5-20) and 0.359 (5-50). MA-cross events are sparser (232 / 130 labels) and, unlike the near-balanced band-touch labels, the class distribution skews — e.g. 5-50 reaches recall_1 = 0.75 with precision_1 = 0.36, i.e. the model just over-predicts class 1. With these features, band-touch signals are far more predictable than MA-cross events.

### Fracdiff factor (mean reversion on the fractionally-differentiated log price)

`fracdiff_side` trades reversion to zero: short when the FFD series z > k·σ, long when z < −k·σ (k = 0.5).

| config | accuracy | recall_1 | f1_0 | f1_1 | n_labeled |
|---|---|---|---|---|---|
| d=0.3 (thres=1e-3) | 0.530 | 0.185 | 0.653 | 0.275 | 934 |
| d=0.5 | 0.525 | 0.129 | 0.661 | 0.205 | 867 |
| d=1.0 | 0.508 | 0.472 | 0.532 | 0.480 | 879 |

- Lower d ⇒ longer memory, smoother z ⇒ profitable trades harder to learn ⇒ the meta-model degenerates to "never trade" (recall_1 0.13–0.19, f1_0 ≈ 0.65).
- d=1.0 is the best fracdiff variant but still slightly behind band-touch signals at matched sample sizes (f1_1 0.480 vs 0.519 for std w=0.5).
- Visualization panel 3 shows the z-series for d=0.3/0.5/1.0 (smooth vs noisy); panel 4 shows raw signal counts per source — frac 16,306 ≫ fracdiff 12,313 ≫ ma_cross 1,457 ≫ std 2σ 187 (log scale).

### Vertical-barrier share & time horizon (label-noise attribution)

Most labels are resolved by the vertical barrier — a coin flip — not by PT/SL. The vbarrier share drops steeply as the barrier lengthens: frac t1=1d → 84.7%, t1=5d → 59.1%, t1=10d → 43.8%.

- frac accuracy stays ~0.52–0.55 across t1; at t1=10d precision_1 falls to 0.42 as more events hit the stop and labels skew toward 0.
- ma_cross 5-20 with t1=5d is the best configuration found (accuracy 0.600, precision_1 0.594) — event-type signals benefit most from a longer holding horizon.
- To cut the vbarrier share without unbalancing the labels, combine a longer t1 with a real profit-take (e.g. ptSl=[0.5,2]) — neither lever alone is enough.

### RF hyperparameter search (time-series CV)

Grid-searched `n_estimators ∈ {50, 100, 200}` × `max_depth ∈ {3, 5, 7}` with `TimeSeriesSplit(3)` scoring F1 on the first 70% of the sample, evaluated on the last 30%. (cqrlib's `PurgedKFold` breaks on DatetimeIndex events, so `TimeSeriesSplit` is used — same future-free property.)

- The best hyperparameters depend on the signal source: frac → 200 trees / depth 7; std w=0.5 → 50 / 7; std w=1.0 → 100 / 5.
- Test f1_1: 0.510 (frac) / 0.535 (std w=0.5) / 0.555 (std w=1.0) — the pattern holds: more labels support deeper/wider forests, while smaller samples make metrics less reliable.

Takeaway: use band width to control signal frequency, and only trust metrics when the test sample is large enough to be statistically meaningful.
